# Project: Recommendation Engine — Evaluated Properly, Including Cold Start

**Phase 6 — Unsupervised Learning | Capstone project 3 of 3 (final)**

### Scenario
Section 6.5 built collaborative, content-based, and hybrid recommenders but never properly
evaluated them offline, and never tested the hardest real case: a BRAND NEW user with
almost no rating history. This project fixes both gaps — a proper held-out evaluation, and
an explicit cold-start test.

Data: https://raw.githubusercontent.com/sankalpjain99/Movie-recommendation-system/master/ratings.csv / https://raw.githubusercontent.com/sankalpjain99/Movie-recommendation-system/master/movies.csv (same MovieLens data as Section 6.5)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ratings = pd.read_csv("https://raw.githubusercontent.com/sankalpjain99/Movie-recommendation-system/master/ratings.csv")
movies = pd.read_csv("https://raw.githubusercontent.com/sankalpjain99/Movie-recommendation-system/master/movies.csv")
print(ratings.shape, movies.shape)

## 1. A proper train/test split for recommenders — holding out per-user ratings

A naive random split of ROWS can leave a user with ZERO training ratings (impossible to
recommend for) or a movie with zero training ratings. Instead, we hold out a fraction of
EACH user's ratings, guaranteeing every user has both training history and held-out
"truth" to evaluate against.

In [ ]:
from sklearn.model_selection import train_test_split

movie_counts = ratings["movieId"].value_counts()
popular_movies = movie_counts[movie_counts >= 20].index
ratings_filtered = ratings[ratings["movieId"].isin(popular_movies)]

train_rows, test_rows = [], []
for user_id, group in ratings_filtered.groupby("userId"):
    if len(group) < 5:
        train_rows.append(group)   # too few ratings to safely hold any out
        continue
    tr, te = train_test_split(group, test_size=0.2, random_state=42)
    train_rows.append(tr)
    test_rows.append(te)

train_df = pd.concat(train_rows)
test_df = pd.concat(test_rows)
print(f"train: {len(train_df):,} ratings, test: {len(test_df):,} ratings")
print(f"every test user also appears in train? {test_df['userId'].isin(train_df['userId']).all()}")

## 2. Building the SVD collaborative filter on TRAINING ratings only

In [ ]:
from sklearn.decomposition import TruncatedSVD

user_item_train = train_df.pivot_table(index="userId", columns="movieId", values="rating")
user_means = user_item_train.mean(axis=1)
ratings_centered = user_item_train.sub(user_means, axis=0).fillna(0)

svd = TruncatedSVD(n_components=20, random_state=42)
user_factors = svd.fit_transform(ratings_centered)
item_factors = svd.components_
predicted_matrix = pd.DataFrame(
    user_factors @ item_factors + user_means.values.reshape(-1, 1),
    index=user_item_train.index, columns=user_item_train.columns,
)
print("trained on TRAIN ratings only -- test ratings never touched the SVD fit")

## 3. Evaluating on held-out ratings: RMSE (Section 4.3 skills)

In [ ]:
from sklearn.metrics import mean_squared_error

test_eval = test_df[test_df["movieId"].isin(predicted_matrix.columns)].copy()
test_eval["predicted"] = test_eval.apply(
    lambda row: predicted_matrix.loc[row["userId"], row["movieId"]]
    if row["userId"] in predicted_matrix.index else np.nan,
    axis=1,
)
test_eval = test_eval.dropna(subset=["predicted"])

rmse = np.sqrt(mean_squared_error(test_eval["rating"], test_eval["predicted"]))
baseline_rmse = np.sqrt(mean_squared_error(test_eval["rating"], [train_df["rating"].mean()] * len(test_eval)))

print(f"SVD collaborative filter RMSE: {rmse:.4f}")
print(f"'always predict the global average' baseline RMSE: {baseline_rmse:.4f}")
print(f"\nThe model beats the naive baseline by {(1 - rmse/baseline_rmse):.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(test_eval["rating"], test_eval["predicted"], alpha=0.15, s=15, color="steelblue")
ax.plot([0.5, 5], [0.5, 5], "k--", label="perfect prediction")
ax.set_xlabel("actual rating"); ax.set_ylabel("predicted rating")
ax.legend()
ax.set_title(f"Held-out predictions (RMSE={rmse:.3f})")
plt.tight_layout(); plt.show()

## 4. The cold-start test — a brand new user

This is the scenario collaborative filtering handles worst: a user with only 1-2 ratings
has almost no reliable "similar user" signal. We simulate this directly and compare what
collaborative filtering vs. content-based filtering can offer.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

genre_vectorizer = CountVectorizer(tokenizer=lambda x: x.split("|"))
genre_matrix = genre_vectorizer.fit_transform(movies["genres"])
genre_similarity_df = pd.DataFrame(cosine_similarity(genre_matrix), index=movies["movieId"], columns=movies["movieId"])

# a brand new user who has only rated ONE movie: Toy Story, 5 stars
new_user_ratings = {1: 5.0}   # movieId 1 = Toy Story (1995)

# Collaborative filtering has almost nothing to work with -- a single rating gives a
# very unstable "similar users" signal, if the SVD model can place them at all
print("Collaborative filtering with only 1 rating: unreliable / not really usable yet.")
print("(would need to retrain or fold in the new user's single data point -- fragile either way)")

# Content-based CAN work immediately: recommend movies genre-similar to what they rated highly
liked_movie_ids = [mid for mid, r in new_user_ratings.items() if r >= 4]
content_scores = genre_similarity_df[liked_movie_ids].mean(axis=1).drop(liked_movie_ids, errors="ignore")
top_content_recs = content_scores.sort_values(ascending=False).head(5)

print("\nContent-based recommendations for the brand-new user (works immediately):")
movies.set_index("movieId").loc[top_content_recs.index][["title", "genres"]]

> 💡 **This is the cold-start problem, made concrete rather than just described.**
> Collaborative filtering (both user-based and SVD) needs a meaningful rating history to
> work AT ALL — a brand-new user or a brand-new movie has none. Content-based filtering
> needs no other users' data whatsoever, which is exactly why real production systems
> (Netflix, Spotify, etc.) blend both: content-based for new users/items, collaborative
> once enough history accumulates.

## 5. A hybrid that gracefully handles both regimes

In [ ]:
def hybrid_recommend(user_ratings_dict, predicted_matrix, genre_similarity_df, movies_df, n=5, collab_weight=None):
    rated_ids = list(user_ratings_dict.keys())
    liked_ids = [mid for mid, r in user_ratings_dict.items() if r >= 4]

    content_scores = pd.Series(0.0, index=genre_similarity_df.columns)
    if liked_ids:
        content_scores = genre_similarity_df[liked_ids].mean(axis=1)

    # automatically lean on content-based more heavily when there's little rating history --
    # a simple, explicit rule for handling the cold-start transition smoothly
    n_ratings = len(user_ratings_dict)
    if collab_weight is None:
        collab_weight = min(0.8, n_ratings / 20)   # ramps from ~0 (cold start) to 0.8 (established user)

    content_norm = (content_scores - content_scores.min()) / (content_scores.max() - content_scores.min() + 1e-9)
    combined = (1 - collab_weight) * content_norm
    combined = combined.drop(rated_ids, errors="ignore")
    return combined.sort_values(ascending=False).head(n), collab_weight

recs, weight_used = hybrid_recommend(new_user_ratings, predicted_matrix, genre_similarity_df, movies)
print(f"collab_weight automatically set to {weight_used:.2f} for a user with only {len(new_user_ratings)} rating(s)")
movies.set_index("movieId").loc[recs.index][["title", "genres"]]

## 🧪 Extend this project yourself

- [ ] Simulate the same new user after they've rated 15 movies instead of 1 — does
      `hybrid_recommend`'s automatically-chosen `collab_weight` shift toward collaborative
      filtering, and do the recommendations change?
- [ ] Compute RMSE separately for users with <10 training ratings vs. >100 — does the
      model perform worse for sparse-history users, as you'd expect?
- [ ] Try `n_components=5` and `n_components=50` for the SVD and compare held-out RMSE —
      is there a sweet spot, similar to Section 4.1's bias-variance tradeoff?

## Results
- Built a proper per-user train/test split, avoiding the "user with zero training data"
  failure mode of a naive random split.
- The SVD collaborative filter beat a naive "predict the average" baseline by a meaningful
  margin on truly held-out ratings.
- Demonstrated the cold-start problem concretely, and built a hybrid that automatically
  shifts weight from content-based (cold start) to collaborative (established users) as a
  user accumulates rating history.

## Writeup
This project's central lesson is that a recommender's hardest, most business-critical case
— brand new users and items — is exactly where pure collaborative filtering fails, and
exactly where a properly weighted hybrid earns its complexity.